# EDA
This notebook contain a EDA of the PDB data and alphaFold,RosetaFold datasets.
The data come from a veriety of places such as the PDB datasest.

The data set containes:
1. Primary Structure - sequence of a chaine of amino acids

    **File** : casp15/deepMeshi14_5_22/data/data30.txt.processed/seq.pt
<br><center><img src="https://upload.wikimedia.org/wikipedia/commons/thumb/3/38/Protein_primary_structure.svg/1280px-Protein_primary_structure.svg.png" width=75%></center> 

2. Secondery structure - Protein secondary structure is the three dimensional form of local segments of proteins. The two most common secondary structural elements are alpha helices and beta sheets, though beta turns and omega loops occur as well. 
 
  * C - Carbon that attached to the carboxly group.
  * Calpha - alpha Carbon, first carbon atom that attaches to a functional group.
  * Cbeta - The second carbon atom.
  * N - Nitrogen.
  * H - Hydrogen, does not exist in the data.
  
<br><center><img src="https://www.researchgate.net/profile/Milos-Rankovic/publication/307606559/figure/fig1/AS:402885302079488@1473066749219/Representation-of-Alanine-amino-acid-Alpha-Carbon-Ca-is-the-so-called-backbone-Carbon.png" width=75%></center> 



  **Files** :  
  * casp15/deepMeshi14_5_22/data/data30.txt.processed/CoordCaNative.pt
  * casp15/deepMeshi14_5_22/data/data30.txt.processed/CoordCbNative.pt
  * casp15/deepMeshi14_5_22/data/data30.txt.processed/CoordCNative.pt 
  * casp15/deepMeshi14_5_22/data/data30.txt.processed/CoordNNative.pt
  
<br><center><img src="https://upload.wikimedia.org/wikipedia/commons/thumb/c/c5/Alpha_beta_structure_%28full%29.png/405px-Alpha_beta_structure_%28full%29.png" width=75%></center> 

Explanation of the diffrent protein structures - https://www.youtube.com/watch?v=MODnIkQvyz0


**Questions:**
1. Were is the distrebution between sources of the Data**
2. GDTTS and IDDTS files?




## Importing libraries

In [6]:
# Pytorch libriaries
import torch 

import numpy as np
import pandas as pd

import os 

# visualization
import plotly.express as px
from tqdm import tqdm


In [2]:
class CFG:
    proteinPath = '/Users/shaharcohen/Studies/MSc/research/DeepPEF/data'
    device = torch.device("cuda:0" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")  # Use GPU is avaliable

## Load datasets

In [4]:
seq_df = torch.load(CFG.proteinPath + '/seq.pt')  # [:200]
ids_df = torch.load(CFG.proteinPath + '/ids.pt')  # [:200]
coordN = torch.load(CFG.proteinPath + '/CoordN.pt')  # [:200]
coordAlpha = torch.load(CFG.proteinPath + '/CoordAlpha.pt')  # [:200]
coordC = torch.load(CFG.proteinPath + '/CoordC.pt')  # [:200]
coordBeta = torch.load(CFG.proteinPath + '/CoordBeta.pt')  # [:200]
nativemask = torch.load(CFG.proteinPath + '/nativemask.pt')  # [:200]
msk = torch.load(CFG.proteinPath + '/mask.pt')  # [:200]
gdtts = torch.load(CFG.proteinPath + '/GDTTS.pt')  # [:200]
iddts = torch.load(CFG.proteinPath + '/IDDTS.pt')  # [:200]
coordNNative = torch.load(CFG.proteinPath + '/CoordNNative.pt')  # [:200]
coordAlphaNative = torch.load(CFG.proteinPath + '/CoordCaNative.pt')  # [:200]
coordCNative = torch.load(CFG.proteinPath + '/CoordCNative.pt')  # [:200]
coordBetaNative = torch.load(CFG.proteinPath + '/CoordCbNative.pt')  # [:200]
embeddings = torch.load(CFG.proteinPath + '/embeddings.pt')  # [:200]

In [4]:
print(f"Number of squences:{len(seq_df)} ")
print(f"Number of rows in a sequence:{len(seq_df[0])} ")
print(f"Number of ids:{len(ids_df)} ")


Number of squences:42380 
Number of rows in a sequence:20 
Number of ids:42380 


Each sequence is represented with 20 rows(one for each amino acid) and the number of columns decided with the length of the sequence.

the mapping of the amino acid order is 

'A': '0', 'C': '1', 'D': '2', 'E': '3', 'F': '4', 'G': '5', 'H': '6', 'I': '7', 'K': '8', 'L': '9',
     'M': '10', 'N': '11', 'P': '12', 'Q': '13', 'R': '14', 'S': '15', 'T': '16', 'V': '17', 'W': '18',
     'Y': '19', '-': '20'

In [5]:
np.array([i.T.shape for i in seq_df])

array([[210,  20],
       [210,  20],
       [ 35,  20],
       ...,
       [ 99,  20],
       [ 99,  20],
       [ 99,  20]])

In [6]:
print(seq_df[0])
print(ids_df[:5])

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 1.,  ..., 0., 0., 0.],
        ...,
        [0., 1., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])
['10#1HF2_1_A__RoseTTAFold', '10#1HF2_1_A__RoseTTAFoldNT', '10#1JB0_12_X__RoseTTAFold', '10#1JB0_12_X__RoseTTAFoldNT', '10#1WAZ_1_A__RoseTTAFold']


**Q: there is no use of the order of the amino acids?**

In [7]:
ids_pr_df = pd.DataFrame()
ids_pr_df["full_name"]= ids_df
ids_pr_df["source"] = ids_pr_df["full_name"].apply(lambda x: x.split("__", 2)[-1])
ids_pr_df["?1"] = ids_pr_df["full_name"].apply(lambda x: x.split("_", 2)[0])
ids_pr_df["?2"] = ids_pr_df["full_name"].apply(lambda x: x.split("_", 2)[1])
ids_pr_df["?3"] = ids_pr_df["full_name"].apply(lambda x: x.split("_", 2)[2])
ids_pr_df["?4"] = ids_pr_df["full_name"].apply(lambda x: x.split("#", 2)[0])


In [8]:
ids_pr_df.head()

,full_name,source,?1,?2,?3,?4
0,10#1HF2_1_A__RoseTTAFold,RoseTTAFold,10#1HF2,1,A__RoseTTAFold,10
1,10#1HF2_1_A__RoseTTAFoldNT,RoseTTAFoldNT,10#1HF2,1,A__RoseTTAFoldNT,10
2,10#1JB0_12_X__RoseTTAFold,RoseTTAFold,10#1JB0,12,X__RoseTTAFold,10
3,10#1JB0_12_X__RoseTTAFoldNT,RoseTTAFoldNT,10#1JB0,12,X__RoseTTAFoldNT,10
4,10#1WAZ_1_A__RoseTTAFold,RoseTTAFold,10#1WAZ,1,A__RoseTTAFold,10


In [9]:
fig = px.histogram(ids_pr_df, ids_pr_df["source"].astype(str), color="source",
                   title="<b>Protein sequences source</b>",
                   labels={"x": "souce",
                           "y": "<b>Data counte</b>"})
fig.show()

**Q:**
- Where is the PDB Native data?
- Where is the mask data comes from
- In the training procedure when using rosettafold data, the training data is duplicate and contaion the same proteins.

In [10]:
# count same proteins
torch.Tensor(seq_df)

ValueError: only one element tensors can be converted to Python scalars

In [ ]:
i=2
torch.all(nativemask[2*i] == nativemask[2*i+1])

tensor(True)

In [ ]:
coordAlpha[2].shape

torch.Size([35, 3])

In [ ]:
len(coordAlpha)

42380

In [ ]:
coordAlpha[2]

tensor([[-2081.6001,   649.7000, -1877.3000],
        [-2103.1001,   702.2000, -1505.5000],
        [-1743.0000,   739.1000, -1351.9000],
        [-1690.4000,  1091.9000, -1261.5000],
        [-1359.4000,  1288.9000, -1315.7000],
        [-1077.1000,  1230.1000, -1062.5000],
        [ -808.6000,  1403.1000, -1270.5000],
        [ -911.8000,  1731.8000, -1177.3000],
        [ -643.6000,  1944.9000, -1036.5000],
        [ -709.5000,  1900.1000,  -679.3000],
        [ -696.8000,  1523.5000,  -708.9000],
        [ -376.2000,  1542.6000,  -903.3000],
        [ -221.5000,  1772.6000,  -654.5000],
        [ -324.8000,  1563.7000,  -361.8000],
        [ -204.8000,  1241.1000,  -511.2000],
        [  119.5000,  1405.7000,  -606.3000],
        [  183.4000,  1519.4000,  -257.2000],
        [   89.3000,  1182.2000,  -121.5000],
        [  322.9000,  1003.6000,  -353.1000],
        [  604.8000,  1241.4000,  -277.3000],
        [  538.0000,  1201.1000,    93.0000],
        [  563.3000,   823.1000,  

In [ ]:
msk

[tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
 tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In the mask file 1 are known and 0 is unknown in the tset

In [19]:
seq_df[0].mean(-1).shape

AttributeError: 'list' object has no attribute 'mean'

In [18]:
seq_df[0].shape

torch.Size([20, 210])

In [ ]:
seq_df[1].shape

torch.Size([20, 210])

In [41]:
counter = 0
for i in msk:
    start_i = torch.where(i)[0]
    end_i = torch.where(i)[-1]
    i = i[start_i[0]:]#end_i[-1]]
    if (torch.any(i==0)):
        counter+=1
        
print(counter)

2104


In [39]:
A1 = coordAlphaNative[0].t()
A2 = coordBetaNative[0].t()
A3 = coordCNative[0].t()
A4 = coordCNative[0].t()
cords_native = torch.stack((A1,A2,A3,A4), dim=1)
cords_native.shape
cords_native.unsqueeze(0).shape

torch.Size([1, 3, 4, 210])

In [38]:
embeddings[0].shape

torch.Size([210, 65])

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")  # Use GPU is avaliable


In [ ]:
import torch
import esm
import numpy as np
import gc
from tqdm import tqdm
import logging

logger = logging.getLogger()
fhandler = logging.FileHandler(filename='mylog.log', mode='a')
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fhandler.setFormatter(formatter)
logger.addHandler(fhandler)
logger.setLevel(logging.DEBUG)
logger.debug("started run")

device = torch.device('cpu')#("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")  # Use GPU is avaliable

def print_gpu():
    print(torch.cuda.get_device_name(0))
    print('Memory Usage:')
    print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
    print('Cached:   ', round(torch.cuda.memory_cached(0)/1024**3,1), 'GB')
    
def get_emb():
    # Load ESM-2 model
    model, alphabet = torch.hub.load("facebookresearch/esm:main", "esm2_t30_150M_UR50D")
    batch_converter = alphabet.get_batch_converter()
    model = model.to(device)
    model.eval()  # disables dropout for deterministic results
    print('Finished loading model.')

    from tqdm import tqdm
    # Prepare data (first 2 sequences from ESMStructuralSplitDataset superfamily / 4)
    all_data = []
    # Using readlines()
    file = open('/Users/shaharcohen/Studies/MSc/research/DeepPEF/data/seq_primar.txt', 'r')
    index = 0
    while True:
        next_line = file.readline()
        if not next_line or index==10:
            break 
        all_data.append([index,next_line])
        index+=1
    seq_emb = []
    batch_size = 2
    for i in tqdm(np.arange(batch_size,len(all_data),batch_size)):
        #print_gpu()
        data = all_data[i-batch_size:i]
        batch_labels, batch_strs, batch_tokens = batch_converter(data)
        batch_lens = (batch_tokens != alphabet.padding_idx).sum(1)
        batch_tokens = batch_tokens.to(device)# move to GPU
        # Extract per-residue representations
        with torch.no_grad():
            results = model(batch_tokens, repr_layers=[30], return_contacts=True)
        token_representations = results["representations"][30]

        # Generate per-sequence representations via averaging
        # NOTE: token 0 is always a beginning-of-sequence token, so the first residue is token 1.
        sequence_representations = []
        for i, tokens_len in enumerate(batch_lens):
            sequence_representations.append(token_representations[i, 1 : tokens_len - 1].mean(0))
        seq_emb.extend(sequence_representations )
        del sequence_representations,token_representations,batch_lens,batch_labels, batch_strs, batch_tokens,results
        torch.cuda.empty_cache()
        gc.collect()
    torch.save(seq_emb,"esm2_t30_150M_UR50D_emb.pt")

def main():
    get_emb()
main()


Using cache found in /Users/shaharcohen/.cache/torch/hub/facebookresearch_esm_main


Finished loading model.


100%|██████████| 4/4 [00:04<00:00,  1.15s/it]


In [9]:
# Move emmbedings to data2:
'''copy emmbeding from data to data2'''
emb_path = "../data/esm2_t12_35M_UR50D/"
emb_new_path = "../data/data2/"
for i in tqdm(np.arange(0,len(seq_df),1)):
    file = torch.load(emb_path+"emb_"+str(i)+".pt")
    torch.save(file,emb_new_path+str(i)+'/'+"emb_esm.pt")

100%|██████████| 42380/42380 [00:21<00:00, 1931.64it/s]
